# FlowSure Data Pipeline — AI-Powered Ticketsysteem

**Vak:** ML Engineering & Ops  
**Inlevering:** Week 6 — Datapipeline onderdeel  
**Team:** [Vul jullie namen in]  

Dit notebook bouwt een end-to-end datapipeline voor het FlowSure klantenondersteuningsproject.
We verwerken twee datasets (Bitext en Twitter) via een medallion-architectuur (Bronze → Silver → Gold)
en demonstreren zowel batch- als streamingverwerking met PySpark op Databricks.

---

# 1. Setup

We importeren alle benodigde PySpark-functies en definiëren herbruikbare helperfuncties
voor datavalidatie, opschonen en opslag.

In [ ]:
from pyspark.sql.functions import (
    col, count, when, isnan, upper, trim, length, lower,
    regexp_replace, regexp_extract, to_timestamp, lit, current_timestamp
)
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql import DataFrame

## 1.1 Helperfuncties

We definiëren herbruikbare functies voor de pipeline.
Dit maakt de code modulair en voorkomt duplicatie tussen batch- en streamingverwerking.

In [ ]:
# --- Constanten ---
VALID_CATEGORIES = [
    "ACCOUNT", "ORDER", "REFUND", "CONTACT", "INVOICE",
    "PAYMENT", "FEEDBACK", "DELIVERY", "SHIPPING",
    "SUBSCRIPTION", "CANCEL"
]

HIGH_PRIORITY_INTENTS = ["complaint", "payment_issue", "cancel_order", "check_cancellation_fee"]
MEDIUM_PRIORITY_INTENTS = ["track_refund", "get_refund", "check_refund_policy", "delivery_period", "registration_problems"]


def save_to_catalog(df: DataFrame, schema: str, table_name: str) -> None:
    """Sla een DataFrame op als Delta-tabel in de Flowsure catalog en print verificatie."""
    full_name = f"flowsure.{schema}.{table_name}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS flowsure.{schema}")
    df.write.mode("overwrite").saveAsTable(full_name)
    row_count = spark.table(full_name).count()
    print(f"{full_name}: {row_count:,} rijen opgeslagen")


def clean_bitext(df: DataFrame) -> DataFrame:
    """Filter kapotte CSV-rijen en schoon Bitext data op.
    
    Verwijdert rijen zonder geldige categorie/intent/instruction.
    Filtert op de 11 bekende UPPERCASE categorieën.
    """
    return (
        df
        .filter(col("instruction").isNotNull())
        .filter(col("intent").isNotNull())
        .filter(trim(col("category")).isin(VALID_CATEGORIES))
        .select(
            trim(col("instruction")).alias("text"),
            trim(col("category")).alias("category"),
            trim(col("intent")).alias("intent"),
            trim(col("response")).alias("response"),
            trim(col("flags")).alias("flags"),
        )
    )


def clean_twitter(df: DataFrame) -> DataFrame:
    """Schoon Twitter data op: filter corrupte rijen, verwijder @mentions,
    converteer timestamps, en maak is_customer boolean.
    """
    spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
    return (
        df
        .filter(col("inbound").isin("True", "False"))
        .withColumn("text_clean", regexp_replace(col("text"), r"@\S+", ""))
        .withColumn("text_clean", regexp_replace(col("text_clean"), r"\s+", " "))
        .withColumn("text_clean", trim(col("text_clean")))
        .withColumn("timestamp", to_timestamp(col("created_at"), "EEE MMM dd HH:mm:ss Z yyyy"))
        .withColumn("is_customer", when(col("inbound") == "True", True).otherwise(False))
        .filter(length(col("text_clean")) > 0)
        .select(
            col("tweet_id"), col("author_id"), col("is_customer"),
            col("timestamp"), col("text_clean").alias("text"),
            col("response_tweet_id"), col("in_response_to_tweet_id"),
        )
    )


def add_priority(df: DataFrame) -> DataFrame:
    """Voeg prioriteitslabel toe op basis van intent."""
    return (
        df
        .withColumn(
            "priority",
            when(col("intent").isin(HIGH_PRIORITY_INTENTS), "high")
            .when(col("intent").isin(MEDIUM_PRIORITY_INTENTS), "medium")
            .otherwise("low")
        )
        .withColumn("text_length", length(col("text")))
    )


def build_conversation_pairs(df: DataFrame) -> DataFrame:
    """Koppel klant-tweets aan support-antwoorden via tweet_id's."""
    df_customers = (
        df.filter(col("is_customer") == True)
        .filter(col("response_tweet_id").isNotNull())
        .withColumn("first_response_id", regexp_extract(col("response_tweet_id"), r"^(\d+)", 1))
        .select(
            col("tweet_id").alias("customer_tweet_id"),
            col("text").alias("customer_text"),
            col("timestamp").alias("customer_timestamp"),
            col("first_response_id"),
        )
    )
    df_support = (
        df.filter(col("is_customer") == False)
        .select(
            col("tweet_id").alias("support_tweet_id"),
            col("text").alias("support_text"),
            col("author_id").alias("company"),
        )
    )
    return (
        df_customers
        .join(df_support, df_customers["first_response_id"] == df_support["support_tweet_id"], "inner")
        .select("customer_tweet_id", "customer_text", "support_text", "company", "customer_timestamp")
    )


def print_catalog_overview():
    """Print een overzicht van alle tabellen in de Flowsure catalog."""
    print("=" * 60)
    print("FLOWSURE DATA PIPELINE — COMPLEET OVERZICHT")
    print("=" * 60)
    for schema in ["bronze", "silver", "gold", "monitoring"]:
        print(f"\n--- flowsure.{schema} ---")
        tables = spark.sql(f"SHOW TABLES IN flowsure.{schema}").collect()
        for t in tables:
            cnt = spark.table(f"flowsure.{schema}.{t.tableName}").count()
            print(f"  {t.tableName}: {cnt:,} rijen")

# 2. Exploratory Data Analysis (EDA)

## 2.1 Bitext Dataset

We laden de Bitext Customer Support dataset (gelabelde klantvragen met intent en categorie).
Deze dataset is bedoeld als trainingsdata voor het **edge-model** (classificatie).

In [ ]:
# Laad Bitext dataset
df_bitext_raw = spark.read.csv(
    "/Volumes/flowsure/default/raw_files/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv",
    header=True,
    inferSchema=True,
)

total = df_bitext_raw.count()
print(f"Bitext: {total:,} rijen, {len(df_bitext_raw.columns)} kolommen")
df_bitext_raw.printSchema()

# Datakwaliteit: hoeveel rijen zijn bruikbaar?
nulls_cat = df_bitext_raw.filter(col("category").isNull()).count()
nulls_int = df_bitext_raw.filter(col("intent").isNull()).count()
clean = df_bitext_raw.filter(
    col("category").isNotNull() & col("intent").isNotNull() & col("instruction").isNotNull()
).count()

print(f"\nNULL in category: {nulls_cat:,} ({nulls_cat/total*100:.1f}%)")
print(f"NULL in intent:   {nulls_int:,} ({nulls_int/total*100:.1f}%)")
print(f"Bruikbare rijen:  {clean:,} ({clean/total*100:.1f}%)")

# Filter op echte categorieën (UPPERCASE) en toon verdeling
df_bitext_valid = df_bitext_raw.filter(
    col("category").isNotNull() & col("intent").isNotNull() & col("instruction").isNotNull()
).filter(trim(col("category")) == upper(trim(col("category"))))

print(f"\nNa filtering op hoofdletters: {df_bitext_valid.count():,} rijen")
print(f"Categorieën: {df_bitext_valid.select('category').distinct().count()}")
print(f"Intents: {df_bitext_valid.select('intent').distinct().count()}")
df_bitext_valid.groupBy("category").count().orderBy("count", ascending=False).show(15, truncate=False)

**Observatie:** Van de 71.963 ruwe rijen zijn er ~45.000 kapotte CSV-artefacten (multiline antwoorden).
Na filtering houden we **26.874 valide rijen** over met **11 categorieën** en **27 intents**.
De verdeling is goed gebalanceerd (~950–1.000 rijen per intent).
ACCOUNT is de grootste categorie (5.986), CANCEL de kleinste (950).

## 2.2 Twitter Dataset

De Twitter Customer Support dataset (~3M tweets) bevat ruwe conversaties zonder labels.
We onderzoeken de structuur, datakwaliteit en of we conversatieparen kunnen reconstrueren.
Bedoeld als trainingsdata voor het **cloud-model** (antwoord genereren).

In [ ]:
# Laad Twitter dataset
df_twitter_raw = spark.read.csv(
    "/Volumes/flowsure/default/raw_files/archive/twcs/twcs.csv",
    header=True,
    inferSchema=True,
)

total_tw = df_twitter_raw.count()
print(f"Twitter: {total_tw:,} rijen, {len(df_twitter_raw.columns)} kolommen")
df_twitter_raw.printSchema()

# Datakwaliteit: hoeveel rijen zijn correct (True/False in inbound)?
correct = df_twitter_raw.filter(col("inbound").isin("True", "False")).count()
corrupt = total_tw - correct
print(f"\nCorrecte rijen:  {correct:,} ({correct/total_tw*100:.1f}%)")
print(f"Corrupte rijen:  {corrupt:,} ({corrupt/total_tw*100:.1f}%)")

# Verdeling klant vs support
print("\n--- Klant vs Support ---")
df_twitter_raw.filter(col("inbound").isin("True", "False")) \
    .groupBy("inbound").count().orderBy("inbound").show()

# Top bedrijven
print("--- Top 5 bedrijven ---")
df_twitter_raw.filter(col("inbound") == "False") \
    .groupBy("author_id").count().orderBy("count", ascending=False).show(5, truncate=False)

# Conversatieparen: hoeveel klant-tweets hebben een antwoord?
customer_tweets = df_twitter_raw.filter(col("inbound") == "True")
with_response = customer_tweets.filter(col("response_tweet_id").isNotNull()).count()
customer_total = customer_tweets.count()
print(f"\nKlant-tweets met support-antwoord: {with_response:,} / {customer_total:,} ({with_response/customer_total*100:.1f}%)")

**Observatie:** 94.8% van de rijen is correct — 5.2% is corrupt door komma's in de tekst.
De `inbound`-kolom bevat naast True/False ook getallen (verschoven kolommen).
108 unieke support-accounts, met AmazonHelp en AppleSupport als grootste.
81% van de klant-tweets heeft een gekoppeld support-antwoord — bruikbaar voor conversatieparen.

## 2.3 EDA Samenvatting

| | Bitext | Twitter |
|---|---|---|
| **Totaal rijen** | 71.963 | 2.966.469 |
| **Bruikbare rijen** | 26.874 (37%) | 2.811.774 (95%) |
| **Vervuild door** | Multiline antwoorden in CSV | Komma's verschuiven kolommen |
| **Categorieën** | 11 (gelabeld) | Geen labels |
| **Intents** | 27 (gelabeld) | Geen labels |
| **Conversatieparen** | Vraag + antwoord per rij | 1.245.030 koppelbare paren |
| **Doel** | Edge-model (classificatie) | Cloud-model (antwoord genereren) |

# 3. Bronze — Ruwe data opslaan

De eerste stap van de medallion-architectuur: de originele data ongewijzigd opslaan als Delta-tabellen.
Delta-format is sneller dan CSV (geen herparsen) en ondersteunt versiebeheer.

In [ ]:
# Sla ruwe data op als Delta-tabellen (Bronze laag)
save_to_catalog(df_bitext_raw, "bronze", "bitext_raw")
save_to_catalog(df_twitter_raw, "bronze", "twitter_raw")

# 4. ETL Pipeline — Bronze → Silver

## 4.1 Bitext opschonen

We filteren de kapotte rijen eruit met de `clean_bitext()` functie:
alleen rijen met een geldige categorie (uit de 11 bekende), intent en instruction.

In [ ]:
# Lees van Bronze en schoon op met de helperfunctie
df_bitext = spark.table("flowsure.bronze.bitext_raw")
df_bitext_silver = clean_bitext(df_bitext)

total_before = df_bitext.count()
total_after = df_bitext_silver.count()
removed = total_before - total_after

print(f"Bronze:    {total_before:,} rijen")
print(f"Silver:    {total_after:,} rijen")
print(f"Verwijderd: {removed:,} rijen ({removed/total_before*100:.1f}%)")
df_bitext_silver.show(3, truncate=70)

**Resultaat:** Van 71.963 ruwe rijen houden we 26.872 schone rijen over (37%).
De 62.7% verwijderde rijen waren CSV-artefacten, geen echte tickets.

## 4.2 Twitter opschonen

We gebruiken de `clean_twitter()` functie: corrupte rijen filteren,
@mentions verwijderen, timestamps converteren, en `is_customer` als boolean.

In [ ]:
# Lees van Bronze en schoon op met de helperfunctie
df_twitter = spark.table("flowsure.bronze.twitter_raw")
df_twitter_silver = clean_twitter(df_twitter)

total_before = df_twitter.count()
total_after = df_twitter_silver.count()

print(f"Bronze:    {total_before:,} rijen")
print(f"Silver:    {total_after:,} rijen")
print(f"Verwijderd: {total_before - total_after:,} rijen")
df_twitter_silver.filter(col("is_customer") == True).show(3, truncate=100)

**Resultaat:** Van 2.966.469 ruwe rijen houden we 2.796.845 over.
@mentions verwijderd, timestamps geconverteerd, `is_customer` als boolean.

### Silver-tabellen opslaan

In [ ]:
save_to_catalog(df_bitext_silver, "silver", "bitext_clean")
save_to_catalog(df_twitter_silver, "silver", "twitter_clean")

# 5. Feature Engineering — Silver → Gold

## 5.1 Gold — Classificatie features (Bitext → Edge-model)

We voegen een prioriteit-label toe met de `add_priority()` functie.
Intents over klachten, annuleringen of betalingsproblemen krijgen een hogere prioriteit.

In [ ]:
# Lees Silver en voeg prioriteit toe
df_bitext_s = spark.table("flowsure.silver.bitext_clean")
df_gold_classification = add_priority(df_bitext_s).select(
    "text", "category", "intent", "priority", "flags", "text_length", "response"
)

print(f"Gold classificatie: {df_gold_classification.count():,} rijen")
print("\n--- Prioriteit verdeling ---")
df_gold_classification.groupBy("priority").count().orderBy("count", ascending=False).show()
df_gold_classification.show(3, truncate=70)

**Resultaat:** 26.872 rijen met prioriteitslabels.
Verdeling: 66.7% low, 18.6% medium, 14.7% high — realistisch voor een support-systeem.

## 5.2 Gold — Conversatieparen (Twitter → Cloud-model)

We koppelen klant-tweets aan support-antwoorden met de `build_conversation_pairs()` functie.
Dit levert vraag-antwoord paren op voor het cloud-model.

In [ ]:
# Lees Silver en bouw conversatieparen
df_twitter_s = spark.table("flowsure.silver.twitter_clean")
df_gold_conversations = build_conversation_pairs(df_twitter_s)

print(f"Gold conversatieparen: {df_gold_conversations.count():,}")
df_gold_conversations.select("customer_text", "support_text", "company").show(3, truncate=80)

### Gold-tabellen opslaan

In [ ]:
save_to_catalog(df_gold_classification, "gold", "classification_features")
save_to_catalog(df_gold_conversations, "gold", "conversation_pairs")

# 6. Streaming Pipeline — Simulatie

We simuleren live ticket-instroom met Spark Structured Streaming.
In plaats van Kafka gebruiken we file-based streaming:
we droppen ruwe data als JSON-bestanden in een map,
en de streaming job pikt ze automatisch op en past
dezelfde opschoonlogica toe als de batch-pipeline.

## 6.1 Simulatiedata klaarzetten

We nemen 1.000 ruwe Twitter-rijen en schrijven ze als 5 JSON-bestanden —
elk bestand simuleert een "batch" nieuwe tickets.

In [ ]:
# Pak 1000 ruwe rijen als simulatiedata
df_sim = (
    spark.table("flowsure.bronze.twitter_raw")
    .filter(col("inbound").isin("True", "False"))
    .limit(1000)
)

streaming_input_path = "/Volumes/flowsure/default/raw_files/streaming_input"
dbutils.fs.rm(streaming_input_path, recurse=True)
dbutils.fs.mkdirs(streaming_input_path)

# repartition(5) = 5 losse bestanden
df_sim.repartition(5).write.mode("overwrite").json(streaming_input_path)

files = dbutils.fs.ls(streaming_input_path)
json_files = [f for f in files if f.name.endswith(".json")]
print(f"{len(json_files)} JSON-bestanden aangemaakt met {df_sim.count():,} rijen")

## 6.2 Streaming job — lezen, opschonen, wegschrijven

We definiëren het schema (vereist voor streaming, anders dan batch waar Spark het kan raden)
en starten een Structured Streaming query die:
1. De JSON-map bewaakt voor nieuwe bestanden
2. Dezelfde opschoonlogica toepast als de batch-pipeline
3. Het resultaat wegschrijft naar een Delta-tabel

We gebruiken `trigger(availableNow=True)` i.p.v. `processingTime` vanwege serverless cluster-beperkingen.

In [ ]:
from pyspark.sql.functions import udf
from datetime import datetime

# Schema voor streaming (identiek aan ruwe Twitter data)
twitter_schema = StructType([
    StructField("tweet_id", StringType(), True),
    StructField("author_id", StringType(), True),
    StructField("inbound", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("text", StringType(), True),
    StructField("response_tweet_id", StringType(), True),
    StructField("in_response_to_tweet_id", StringType(), True),
])

checkpoint_path = "/Volumes/flowsure/default/raw_files/streaming_checkpoint"
dbutils.fs.rm(checkpoint_path, recurse=True)

# UDF voor timestamp-parsing (to_timestamp werkt niet in streaming met dit formaat)
@udf(TimestampType())
def parse_twitter_ts(ts_string):
    if ts_string is None:
        return None
    try:
        return datetime.strptime(ts_string, "%a %b %d %H:%M:%S %z %Y")
    except Exception:
        return None

# Lees de map als stream
df_stream = spark.readStream.schema(twitter_schema).json(streaming_input_path)

# Zelfde opschoonlogica als batch, maar met UDF voor timestamp
df_stream_clean = (
    df_stream
    .filter(col("inbound").isin("True", "False"))
    .withColumn("text_clean", regexp_replace(col("text"), r"@\S+", ""))
    .withColumn("text_clean", regexp_replace(col("text_clean"), r"\s+", " "))
    .withColumn("text_clean", trim(col("text_clean")))
    .withColumn("timestamp", parse_twitter_ts(col("created_at")))
    .withColumn("is_customer", when(col("inbound") == "True", True).otherwise(False))
    .withColumn("processed_at", current_timestamp())
    .filter(length(col("text_clean")) > 0)
    .select(
        col("tweet_id"), col("author_id"), col("is_customer"),
        col("timestamp"), col("text_clean").alias("text"), col("processed_at"),
    )
)

# Schrijf naar Delta-tabel
query = (
    df_stream_clean.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("flowsure.gold.streaming_processed")
)
query.awaitTermination()

# Verificatie
df_streamed = spark.table("flowsure.gold.streaming_processed")
print(f"Streaming verwerkt: {df_streamed.count():,} rijen")
df_streamed.show(3, truncate=80)

**Resultaat:** 994 van de 1.000 rijen verwerkt (6 weggevallen door lege tekst).
De `processed_at` kolom toont het verwerkingstijdstip.

In productie zou de `streaming_input` map vervangen worden door Kafka,
de job periodiek gescheduled worden via Databricks Workflows,
en de checkpoint zorgt ervoor dat alleen nieuwe bestanden worden verwerkt.

# 7. Data Monitoring — Drift Baseline

We leggen de verdeling van de trainingsdata vast als referentiepunt.
Als toekomstige data significant afwijkt (bijv. plotseling 40% PAYMENT i.p.v. 7%),
is dat een drift-signaal en moet het model hertraind worden.

In [ ]:
# Baseline statistieken berekenen en opslaan
df_gold = spark.table("flowsure.gold.classification_features")
total = df_gold.count()

print(f"=== DRIFT BASELINE — {total:,} rijen ===")

# Categorie baseline
print("\n--- Categorie verdeling ---")
df_cat_baseline = (
    df_gold.groupBy("category").count()
    .withColumn("percentage", (col("count") / total * 100).cast("decimal(5,1)"))
    .withColumn("baseline_date", current_timestamp())
    .orderBy("count", ascending=False)
)
df_cat_baseline.select("category", "count", "percentage").show(15, truncate=False)

# Prioriteit baseline
print("--- Prioriteit verdeling ---")
df_prio_baseline = (
    df_gold.groupBy("priority").count()
    .withColumn("percentage", (col("count") / total * 100).cast("decimal(5,1)"))
    .withColumn("baseline_date", current_timestamp())
)
df_prio_baseline.select("priority", "count", "percentage").show()

# Tekstlengte statistieken
print("--- Tekstlengte statistieken ---")
df_gold.select("text_length").summary("mean", "stddev", "min", "50%", "max").show()

# Opslaan als Delta-tabellen
save_to_catalog(df_cat_baseline, "monitoring", "baseline_categories")
save_to_catalog(df_prio_baseline, "monitoring", "baseline_priorities")

**Resultaat:** ACCOUNT is de dominante categorie (22.3%), twee derde is low-priority.
Gemiddelde tekstlengte ~47 tekens. Baselines opgeslagen in `flowsure.monitoring`.

# 8. Eindoverzicht

In [ ]:
print_catalog_overview()

# 9. Conclusie & Volgende Stappen

## Wat we hebben gebouwd

Een end-to-end datapipeline die twee ruwe datasets transformeert naar modelklare output,
met een medallion-architectuur (Bronze → Silver → Gold) op Databricks.

| Laag | Tabel | Rijen | Doel |
|---|---|---|---|
| Bronze | bitext_raw | 71.963 | Ruwe CSV, ongewijzigd |
| Bronze | twitter_raw | 2.966.469 | Ruwe CSV, ongewijzigd |
| Silver | bitext_clean | 26.872 | Gevalideerd, kapotte rijen verwijderd |
| Silver | twitter_clean | 2.796.845 | Corrupte rijen gefilterd, tekst opgeschoond, timestamps geconverteerd |
| Gold | classification_features | 26.872 | Intent + categorie + prioriteit → edge-model |
| Gold | conversation_pairs | 1.078.425 | Vraag-antwoord paren → cloud-model |
| Gold | streaming_processed | 994 | Live tickets verwerkt via Structured Streaming |
| Monitoring | baseline_categories | 11 | Referentieverdeling voor drift detectie |
| Monitoring | baseline_priorities | 3 | Referentieverdeling voor drift detectie |

## Leerdoelen gedekt

- **Leerdoel 1** — End-to-end datapipeline: CSV → validatie → transformatie → modelklare output
- **Leerdoel 2** — Batch (3M rijen met Spark) én streaming (Structured Streaming met trigger)
- **Leerdoel 5** — Drift baseline vastgelegd voor continue monitoring